In [7]:
!pip install tensorflow

Defaulting to user installation because normal site-packages is not writeable
  Using cached tensorflow-2.20.0-cp313-cp313-win_amd64.whl.metadata (4.6 kB)
  Using cached absl_py-2.3.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached astunparse-1.6.3-py2.py3-none-any.whl.metadata (4.4 kB)
  Using cached google_pasta-0.2.0-py3-none-any.whl.metadata (814 bytes)
  Using cached libclang-18.1.1-py2.py3-none-win_amd64.whl.metadata (5.3 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached grpcio-1.76.0-cp313-cp313-win_amd64.whl.metadata (3.8 kB)
  Using cached tensorboard-2.20.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached h5py-3.15.1-cp313-cp313-win_amd64.whl.metadata (3.1 kB)
   ---------------------------------------- 0.0/332.0 MB ? eta -:--:--
   ---------------------------------------- 2.6/332.0 MB 14.1 MB/s eta 0:00:24
    --------------------------------------- 5.8/332.0 MB 14.6 MB/s eta 0:00:23
   - -------------------------------------- 9.2/332.0

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

In [13]:
# ============================================================================
# 1. LOAD AND PREPARE DATA
# ============================================================================
print("="*60)
print("PURE NEURAL NETWORK - TASTE RATING PREDICTION")
print("="*60)

# Load your data
df = pd.read_csv("cocktail_dataset_all.csv")

# Extract features and target
ingredient_cols = [col for col in df.columns if '_pct' in col]
print(f"Number of ingredient features: {len(ingredient_cols)}")

# Use ONLY taste_rating as target
target_col = 'taste_rating'

# Remove rows with missing taste ratings
data = df.dropna(subset=[target_col])
print(f"Total samples with taste ratings: {len(data)}")

X = data[ingredient_cols].values  # Raw ingredient percentages only
y = data[target_col].values.reshape(-1, 1)  # Reshape for neural network

print(f"\nData shape: X={X.shape}, y={y.shape}")
print(f"Taste rating range: {y.min():.1f} to {y.max():.1f}")
print(f"Mean taste rating: {y.mean():.2f} ± {y.std():.2f}")

PURE NEURAL NETWORK - TASTE RATING PREDICTION
Number of ingredient features: 240
Total samples with taste ratings: 4603

Data shape: X=(4603, 240), y=(4603, 1)
Taste rating range: 1.0 to 10.0
Mean taste rating: 6.41 ± 1.09


In [3]:
# ============================================================================
# 2. SPLIT DATA
# ============================================================================
print("\n" + "="*60)
print("SPLITTING DATA")
print("="*60)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")


SPLITTING DATA
Training set: 3682 samples
Test set: 921 samples


In [4]:
# ============================================================================
# 3. SCALE DATA (CRITICAL FOR NEURAL NETWORKS)
# ============================================================================
print("\n" + "="*60)
print("SCALING DATA")
print("="*60)

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train)
X_test_scaled = scaler_X.transform(X_test)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled = scaler_y.transform(y_test)

print("Data scaling complete")
print(f"X scaled: mean={X_train_scaled.mean():.3f}, std={X_train_scaled.std():.3f}")
print(f"y scaled: mean={y_train_scaled.mean():.3f}, std={y_train_scaled.std():.3f}")


SCALING DATA
Data scaling complete
X scaled: mean=0.000, std=1.000
y scaled: mean=-0.000, std=1.000


In [8]:
# ============================================================================
# CORRECT APPROACH: MULTI-CLASS CLASSIFICATION
# ============================================================================
import tensorflow as tf
import numpy as np

# Convert ratings to integer classes (1-10)
y_train_int = y_train.astype(int) - 1  # Convert to 0-9 for classification
y_test_int = y_test.astype(int) - 1

print(f"Unique ratings in data: {np.unique(y_train)}")
print(f"Converted to classes: {np.unique(y_train_int)}")

# Build classification network
def build_classification_nn(input_dim, num_classes=10):
    """Neural network for rating classification (1-10)"""
    model = keras.models.Sequential([
        layers.Input(shape=(input_dim,)),
        
        # Hidden layers
        layers.Dense(64, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        
        layers.Dense(32, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        
        # Output layer: 10 classes with softmax
        layers.Dense(num_classes, activation='softmax')
    ])
    
    # Use categorical crossentropy for classification
    model.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',  # For integer labels
        metrics=['accuracy', 
                tf.keras.metrics.TopKCategoricalAccuracy(k=2, name='top2_accuracy'),
                tf.keras.metrics.TopKCategoricalAccuracy(k=3, name='top3_accuracy')]
    )
    
    return model

# Build and train
class_model = build_classification_nn(input_dim=X_train_scaled.shape[1], num_classes=10)
class_model.summary()



Unique ratings in data: [ 1  2  3  4  5  6  7  8  9 10]
Converted to classes: [0 1 2 3 4 5 6 7 8 9]


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 64)             │        15,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 10)             │           330 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,218 (71.16 KB)

 Trainable params: 18,026 (70.41 KB)

 Non-trainable params: 192 (768.00 B)

In [9]:
# Train
history = class_model.fit(
    X_train_scaled, y_train_int,
    validation_split=0.2,
    epochs=100,
    batch_size=32,
    # callbacks=[
    #     tf.keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True),
    #     tf.keras.callbacks.ReduceLROnPlateau(factor=0.5, patience=10)
    # ],
    verbose=1
)

Epoch 1/100
93/93 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.1440 - loss: 2.6894 - top2_accuracy: 0.1528 - top3_accuracy: 0.2482 - val_accuracy: 0.2510 - val_loss: 2.1348 - val_top2_accuracy: 0.0611 - val_top3_accuracy: 0.1153
Epoch 2/100
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.2859 - loss: 2.0898 - top2_accuracy: 0.0927 - top3_accuracy: 0.1705 - val_accuracy: 0.3569 - val_loss: 1.8467 - val_top2_accuracy: 0.0190 - val_top3_accuracy: 0.0421
Epoch 3/100
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.4109 - loss: 1.7210 - top2_accuracy: 0.0357 - top3_accuracy: 0.0856 - val_accuracy: 0.3921 - val_loss: 1.6372 - val_top2_accuracy: 0.0054 - val_top3_accuracy: 0.0163
Epoch 4/100
93/93 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.4699 - loss: 1.4992 - top2_accuracy: 0.0166 - top3_accuracy: 0.0431 - val_accuracy: 0.4098 - val_loss: 1.5141 - val_top2_accuracy: 0.0014 - val_top3_accuracy: 0.0054
Epoch 5/100
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5039 - loss

In [11]:
# Predict classes
y_pred_proba = class_model.predict(X_test_scaled)
y_pred_classes = np.argmax(y_pred_proba, axis=1) + 1  # Convert back to 1-10
y_test_original = y_test.copy().flatten()

# Calculate metrics for CLASSIFICATION
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# 1. Exact match accuracy
exact_accuracy = accuracy_score(y_test_original, y_pred_classes)
print(f"Exact match accuracy: {exact_accuracy:.3f}")

# 2. Within ±1 point accuracy
within_1 = np.mean(np.abs(y_test_original - y_pred_classes) <= 1)
print(f"Within ±1 point accuracy: {within_1:.3f}")

# 3. Top-2 accuracy (predicts correct or adjacent rating)
top2_correct = 0
for i in range(len(y_test_original)):
    top2_preds = np.argsort(y_pred_proba[i])[-2:] + 1  # Top 2 predictions
    if y_test_original[i] in top2_preds:
        top2_correct += 1
top2_accuracy = top2_correct / len(y_test_original)
print(f"Top-2 accuracy: {top2_accuracy:.3f}")

# 4. Mean absolute error (still useful)
mae_classification = np.mean(np.abs(y_test_original - y_pred_classes))
print(f"MAE (classification): {mae_classification:.3f}")

29/29 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step 
Exact match accuracy: 0.453
Within ±1 point accuracy: 0.885
Top-2 accuracy: 0.745
MAE (classification): 0.695


In [15]:
# ============================================================================
# 4. BUILD NEURAL NETWORK ARCHITECTURE
# ============================================================================
print("\n" + "="*60)
print("BUILDING NEURAL NETWORK")
print("="*60)

def build_small_taste_predictor(input_dim):
    inputs = layers.Input(shape=(input_dim,))
    
    # Layer 1: Small embedding
    x = layers.Dense(64, activation='relu', 
                     kernel_regularizer=keras.regularizers.l2(0.01))(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)  # HIGH dropout for small data
    
    # Layer 2: Even smaller
    x = layers.Dense(32, activation='relu',
                     kernel_regularizer=keras.regularizers.l2(0.01))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    
    # Layer 3: Tiny
    x = layers.Dense(16, activation='relu')(x)
    
    # Output
    outputs = layers.Dense(1, activation='linear')(x)
    
    model = keras.Model(inputs=inputs, outputs=outputs)
    
    optimizer = keras.optimizers.Adam(learning_rate=0.001)
    
    model.compile(
        optimizer=optimizer,
        loss='mse',
        metrics=['mae']
    )
    
    return model

# Build and check
small_model = build_small_taste_predictor(input_dim)
small_model.summary()


BUILDING NEURAL NETWORK


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_3 (InputLayer)      │ (None, 240)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_16 (Dense)                │ (None, 64)             │        15,424 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_8           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_6 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_17 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_9           │ (None, 32)             │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_18 (Dense)                │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_19 (Dense)                │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,433 (72.00 KB)

 Trainable params: 18,241 (71.25 KB)

 Non-trainable params: 192 (768.00 B)

In [26]:
# ============================================================================
# 5. TRAIN THE MODEL
# ============================================================================
print("\n" + "="*60)
print("TRAINING NEURAL NETWORK")
print("="*60)

# Callbacks for better training
callbacks_list = [
    callbacks.EarlyStopping(
        monitor='val_loss',
        patience=30,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=15,
        min_lr=1e-6,
        verbose=1
    ),
    callbacks.ModelCheckpoint(
        'best_taste_model.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    )
]

# Train model
print("Starting training...")
history = small_model.fit(
    X_train_scaled, y_train_scaled,
    validation_split=0.2,
    epochs=300,
    batch_size=32,
    # callbacks=callbacks_list,
    verbose=1
)


TRAINING NEURAL NETWORK
Starting training...
Epoch 1/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2016 - mae: 0.2853 - val_loss: 0.7899 - val_mae: 0.6343
Epoch 2/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1944 - mae: 0.2785 - val_loss: 0.7903 - val_mae: 0.6343
Epoch 3/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1964 - mae: 0.2765 - val_loss: 0.7903 - val_mae: 0.6344
Epoch 4/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2012 - mae: 0.2839 - val_loss: 0.7902 - val_mae: 0.6344
Epoch 5/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1925 - mae: 0.2790 - val_loss: 0.7894 - val_mae: 0.6340
Epoch 6/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1972 - mae: 0.2801 - val_loss: 0.7893 - val_mae: 0.6338
Epoch 7/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.1968 - mae: 0.2771 - val_loss: 0.7893 - val_mae: 0.6339
Epoch 8/300
93/93 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - loss: 0.2085 - mae: 0.2887 - val_loss: 0.7902 - val_mae: 0.6345
Epoch 9/300
93/93 

In [27]:
# ============================================================================
# 6. EVALUATE ON TEST SET
# ============================================================================
print("\n" + "="*60)
print("EVALUATION ON TEST SET")
print("="*60)

# Predict on test set
y_pred_scaled = small_model.predict(X_test_scaled, verbose=0)

# Inverse transform to original scale
y_pred = scaler_y.inverse_transform(y_pred_scaled).flatten()
y_test_original = scaler_y.inverse_transform(y_test_scaled).flatten()

# Calculate metrics
mse = mean_squared_error(y_test_original, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test_original, y_pred)
r2 = r2_score(y_test_original, y_pred)

print("\nPERFORMANCE METRICS:")
print("-" * 40)
print(f"Mean Squared Error (MSE): {mse:.4f}")
print(f"Root Mean Squared Error (RMSE): {rmse:.3f}")
print(f"Mean Absolute Error (MAE): {mae:.3f}")
print(f"R² Score: {r2:.3f}")
print(f"Standard Deviation of Errors: {np.std(y_test_original - y_pred):.3f}")

# Baseline comparison (predicting mean)
baseline_pred = np.full_like(y_test_original, y_train.mean())
baseline_r2 = r2_score(y_test_original, baseline_pred)
print(f"\nBaseline (predicting mean): R² = {baseline_r2:.3f}")
print(f"Model improvement over baseline: {r2 - baseline_r2:.3f}")


EVALUATION ON TEST SET

PERFORMANCE METRICS:
----------------------------------------
Mean Squared Error (MSE): 0.8284
Root Mean Squared Error (RMSE): 0.910
Mean Absolute Error (MAE): 0.684
R² Score: 0.260
Standard Deviation of Errors: 0.910

Baseline (predicting mean): R² = -0.000
Model improvement over baseline: 0.260


In [ ]:

# ============================================================================
# 7. CONFUSION MATRIX FOR TASTE RATING
# ============================================================================
print("\n" + "="*60)
print("CONFUSION MATRIX (Binned Ratings)")
print("="*60)

# Bin ratings into categories for confusion matrix
def bin_ratings(ratings):
    """Convert continuous ratings to binned categories"""
    bins = [0, 4, 5, 6, 7, 8, 10.1]  # 6 bins
    labels = ['Poor (1-4)', 'Below Avg (4-5)', 'Average (5-6)', 
              'Good (6-7)', 'Very Good (7-8)', 'Excellent (8-10)']
    return pd.cut(ratings, bins=bins, labels=labels, right=False)

# Bin actual and predicted ratings
y_test_binned = bin_ratings(y_test_original)
y_pred_binned = bin_ratings(y_pred)

# Create confusion matrix
from sklearn.metrics import confusion_matrix, classification_report

cm = confusion_matrix(y_test_binned, y_pred_binned, 
                      labels=y_test_binned.cat.categories)

print("\nConfusion Matrix (Actual vs Predicted Categories):")
print("-" * 60)

# Display confusion matrix
cm_df = pd.DataFrame(cm, 
                     index=y_test_binned.cat.categories,
                     columns=y_test_binned.cat.categories)

plt.figure(figsize=(10, 8))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=True,
            square=True, linewidths=0.5, linecolor='gray')
plt.title('Confusion Matrix - Taste Rating Predictions', fontsize=14, pad=20)
plt.xlabel('Predicted Rating Category', fontsize=12)
plt.ylabel('Actual Rating Category', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('taste_rating_confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

# Classification report
print("\nCLASSIFICATION REPORT:")
print("-" * 60)
print(classification_report(y_test_binned, y_pred_binned, 
                            target_names=y_test_binned.cat.categories))

# Calculate accuracy within tolerance
tolerance = 1.0  # Within 1 point
within_tolerance = np.abs(y_test_original - y_pred) <= tolerance
accuracy_within_1 = np.mean(within_tolerance) * 100

print(f"\nPredictions within ±{tolerance} point: {accuracy_within_1:.1f}%")

# ============================================================================
# 8. VISUALIZE RESULTS
# ============================================================================
print("\n" + "="*60)
print("VISUALIZATIONS")
print("="*60)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

# Plot 1: Training history
axes[0, 0].plot(history.history['loss'], label='Training Loss', linewidth=2)
axes[0, 0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Loss (MSE)')
axes[0, 0].set_title('Training History')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Actual vs Predicted scatter
axes[0, 1].scatter(y_test_original, y_pred, alpha=0.6, edgecolors='w', linewidth=0.5)
axes[0, 1].plot([y_test_original.min(), y_test_original.max()], 
                [y_test_original.min(), y_test_original.max()], 
                'r--', linewidth=2, label='Perfect Prediction')
axes[0, 1].set_xlabel('Actual Taste Rating')
axes[0, 1].set_ylabel('Predicted Taste Rating')
axes[0, 1].set_title(f'Taste Rating Predictions (R² = {r2:.3f})')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Plot 3: Residual plot
residuals = y_test_original - y_pred
axes[0, 2].scatter(y_pred, residuals, alpha=0.6, edgecolors='w', linewidth=0.5)
axes[0, 2].axhline(y=0, color='r', linestyle='--', linewidth=2)
axes[0, 2].set_xlabel('Predicted Taste Rating')
axes[0, 2].set_ylabel('Residuals (Actual - Predicted)')
axes[0, 2].set_title('Residual Plot')
axes[0, 2].grid(True, alpha=0.3)

# Plot 4: Error distribution
axes[1, 0].hist(residuals, bins=30, edgecolor='black', alpha=0.7)
axes[1, 0].axvline(x=0, color='r', linestyle='--', linewidth=2, label='Zero Error')
axes[1, 0].set_xlabel('Prediction Error')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title(f'Error Distribution (Mean = {residuals.mean():.2f})')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Plot 5: Prediction accuracy by rating level
rating_levels = np.arange(1, 11)
accuracy_by_level = []
for level in rating_levels:
    mask = (y_test_original >= level - 0.5) & (y_test_original < level + 0.5)
    if mask.any():
        level_errors = np.abs(y_test_original[mask] - y_pred[mask])
        accuracy = np.mean(level_errors <= 1.0)
        accuracy_by_level.append(accuracy)
    else:
        accuracy_by_level.append(np.nan)

axes[1, 1].bar(rating_levels[:len(accuracy_by_level)], accuracy_by_level, alpha=0.7)
axes[1, 1].set_xlabel('Actual Rating Level')
axes[1, 1].set_ylabel('Accuracy (within ±1 point)')
axes[1, 1].set_title('Prediction Accuracy by Rating Level')
axes[1, 1].set_xticks(rating_levels)
axes[1, 1].set_ylim(0, 1)
axes[1, 1].grid(True, alpha=0.3)

# Plot 6: Feature importance (using gradient-based importance)
def compute_feature_importance(model, X_sample, feature_names):
    """Compute approximate feature importance using gradients"""
    X_tensor = tf.convert_to_tensor(X_sample, dtype=tf.float32)
    
    with tf.GradientTape() as tape:
        tape.watch(X_tensor)
        predictions = model(X_tensor)
    
    gradients = tape.gradient(predictions, X_tensor)
    importance = tf.reduce_mean(tf.abs(gradients), axis=0).numpy()
    
    # Get top 10 features
    top_idx = np.argsort(importance)[-10:][::-1]
    
    return importance, top_idx

# Compute on a sample of test data
sample_idx = np.random.choice(len(X_test_scaled), size=100, replace=False)
X_sample = X_test_scaled[sample_idx]
importance, top_idx = compute_feature_importance(model, X_sample, ingredient_cols)

top_features = [ingredient_cols[i] for i in top_idx]
top_importance = importance[top_idx]

axes[1, 2].barh(range(len(top_features)), top_importance)
axes[1, 2].set_yticks(range(len(top_features)))
axes[1, 2].set_yticklabels([f[:15] for f in top_features])
axes[1, 2].set_xlabel('Gradient Importance')
axes[1, 2].set_title('Top 10 Important Ingredients (NN learned)')
axes[1, 2].invert_yaxis()

plt.tight_layout()
plt.savefig('neural_network_results.png', dpi=300, bbox_inches='tight')
plt.show()

# ============================================================================
# 9. ANALYZE WHAT THE NETWORK LEARNED
# ============================================================================
print("\n" + "="*60)
print("WHAT THE NEURAL NETWORK LEARNED")
print("="*60)

print("\nTop 10 Most Influential Ingredients (via gradients):")
for i, (feature, imp) in enumerate(zip(top_features, top_importance), 1):
    print(f"{i:2}. {feature:<30}: {imp:.4f}")

# Analyze predictions
print(f"\nPrediction Analysis:")
print(f"- {'Correct predictions (within ±0.5)':<40}: {(np.abs(residuals) <= 0.5).sum():>4} samples")
print(f"- {'Acceptable predictions (within ±1.0)':<40}: {(np.abs(residuals) <= 1.0).sum():>4} samples")
print(f"- {'Large errors (>2.0)':<40}: {(np.abs(residuals) > 2.0).sum():>4} samples")

# Analyze by rating value
print(f"\nPerformance by Rating Range:")
rating_ranges = [(1, 4), (4, 6), (6, 8), (8, 10)]
for r_min, r_max in rating_ranges:
    mask = (y_test_original >= r_min) & (y_test_original < r_max)
    if mask.any():
        range_r2 = r2_score(y_test_original[mask], y_pred[mask])
        range_mae = mean_absolute_error(y_test_original[mask], y_pred[mask])
        print(f"  Ratings {r_min}-{r_max}: R² = {range_r2:.3f}, MAE = {range_mae:.2f}")

# ============================================================================
# 10. SAVE MODEL AND RESULTS
# ============================================================================
print("\n" + "="*60)
print("SAVING MODEL")
print("="*60)

# Save the trained model
model.save('taste_rating_predictor.keras')
print("✓ Model saved as 'taste_rating_predictor.keras'")

# Save scalers
import joblib
joblib.dump(scaler_X, 'feature_scaler.pkl')
joblib.dump(scaler_y, 'target_scaler.pkl')
print("✓ Scalers saved")

# Save performance metrics
results = {
    'r2_score': r2,
    'rmse': rmse,
    'mae': mae,
    'accuracy_within_1': accuracy_within_1,
    'feature_names': ingredient_cols,
    'best_epoch': len(history.history['loss'])
}
joblib.dump(results, 'model_performance.pkl')
print("✓ Performance metrics saved")

# ============================================================================
# 11. PREDICTION FUNCTION FOR NEW RECIPES
# ============================================================================
print("\n" + "="*60)
print("PREDICTION FUNCTION")
print("="*60)

def predict_taste_rating(ingredient_percentages, model, scaler_X, scaler_y):
    """
    Predict taste rating for a new cocktail recipe.
    
    Parameters:
    - ingredient_percentages: List or array of 244 ingredient percentages
    - model: Trained neural network
    - scaler_X: Fitted feature scaler
    - scaler_y: Fitted target scaler
    
    Returns:
    - Predicted taste rating (1-10 scale)
    """
    # Convert to array and reshape
    X_new = np.array(ingredient_percentages).reshape(1, -1)
    
    # Scale features
    X_new_scaled = scaler_X.transform(X_new)
    
    # Predict
    y_pred_scaled = model.predict(X_new_scaled, verbose=0)
    
    # Inverse transform to original scale
    y_pred = scaler_y.inverse_transform(y_pred_scaled)[0, 0]
    
    # Clip to valid range
    y_pred = np.clip(y_pred, 1, 10)
    
    return y_pred

# Test with a sample recipe
print("\nSample Prediction:")
sample_recipe = np.zeros(len(ingredient_cols))
# Set some example values
if 'gin_pct' in ingredient_cols:
    idx = ingredient_cols.index('gin_pct')
    sample_recipe[idx] = 40.0
if 'lime_juice_pct' in ingredient_cols:
    idx = ingredient_cols.index('lime_juice_pct')
    sample_recipe[idx] = 25.0
if 'simple_syrup_pct' in ingredient_cols:
    idx = ingredient_cols.index('simple_syrup_pct')
    sample_recipe[idx] = 15.0

predicted_taste = predict_taste_rating(sample_recipe, model, scaler_X, scaler_y)
print(f"Predicted taste rating: {predicted_taste:.1f}/10")

# ============================================================================
# 12. FINAL SUMMARY
# ============================================================================
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)

print(f"\nModel Performance Summary:")
print(f"  R² Score: {r2:.3f}")
print(f"  RMSE: {rmse:.3f} points")
print(f"  MAE: {mae:.3f} points")
print(f"  Accuracy within ±1 point: {accuracy_within_1:.1f}%")

if r2 > 0.5:
    print(f"\n✅ EXCELLENT! Model is ready for Grammatical Evolution")
elif r2 > 0.4:
    print(f"\n⚠️  ACCEPTABLE - May work for GE with careful tuning")
elif r2 > 0.3:
    print(f"\n⚠️  MARGINAL - Consider adding interaction features")
else:
    print(f"\n❌ POOR - Neural network failed to learn from raw data")

print(f"\nNext steps for Grammatical Evolution:")
print("1. Use 'predict_taste_rating()' function as fitness function")
print("2. Monitor that predictions make sense (check sample recipes)")
print("3. If R² < 0.4, consider adding engineered features")

print(f"\nFiles created:")
print("  - taste_rating_predictor.keras (trained model)")
print("  - feature_scaler.pkl, target_scaler.pkl")
print("  - model_performance.pkl (metrics)")
print("  - taste_rating_confusion_matrix.png")
print("  - neural_network_results.png")